# Liquidity Absorption A06 — Deribit derivatives context around entries (−240 … +240 min)

Protocol: `discussions/Liquidity_Absorption/023_claude_a06_deribit_entry_context_protocol.md`. Engine: `src/data_engineering/modeling/liquidity_absorption_stage_a06.py`.

- **Anchor:** the frozen A04 entry (first post-signal Binance print) floored to the minute; minute 0 = entry minute.
- **Signs:** everything is signed to the trade side, so positive = favourable to the trade.
- **Deribit-only information**, from the downloaded public trade history:
  - perpetual: liquidations by side, mark–index premium, block trades, taker imbalance and price path;
  - options: short-dated ATM IV, put–call skew, taker-signed option delta flow, blocks;
  - nearest dated-future basis.
- **Groups:** TARGET / STOP / TIMEOUT, plus WIN / LOSS by gross sign, and 3 hour-matched non-signal **CONTROL** windows per event.
- **Status:** in-sample and outcome-conditioned. Pre-entry differences that survive BH q ≤ 0.10 are holdout candidates only.

Environment: `LA06_SYNTHETIC=1` for a smoke run.

In [ ]:
from pathlib import Path
import json, os, sys, tempfile, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 250)
load_dotenv()
REPO = Path.cwd().resolve()
while REPO != REPO.parent and not (REPO / 'src' / 'data_engineering').exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'src'))
from data_engineering.modeling import liquidity_absorption_stage_a06 as a6
SYNTHETIC = os.getenv('LA06_SYNTHETIC', '').strip() == '1'
CFG = a6.Config()
if SYNTHETIC:
    TMP = Path(tempfile.mkdtemp(prefix='la06_')); HIST = TMP / 'history'; OUT = TMP / 'out'
else:
    HIST = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve() / 'history'
    OUT = REPO / 'notebooks' / 'Liquidity_Absorption' / '06'
OUT.mkdir(parents=True, exist_ok=True)
print(a6.VERSION, '| synthetic' if SYNTHETIC else '', '| history', HIST, '| out', OUT)
print(json.dumps(a6.registration(CFG), indent=1, default=str))

## 1. Cohort (frozen A04 entries) and hour-matched controls

In [ ]:
COH = a6.load_cohort(REPO)
if SYNTHETIC:   # shift the real cohort onto a short synthetic calendar
    days_s = [str(x.date()) for x in pd.date_range('2026-07-01', periods=16, freq='D')]
    COH = COH.head(120).copy()
    base = pd.Timestamp('2026-07-02', tz='UTC').value // 10**6
    COH['entry_ms'] = base + (np.arange(len(COH)) % 13) * 86_400_000 + (COH.entry_ms % 86_400_000)
    CFG = a6.Config(first_day=days_s[0], last_day=days_s[-1])
    tg = COH[COH.group.eq('TARGET')]
    for cur in ('BTC', 'ETH'):
        sub = tg[tg.currency.eq(cur)]
        a6.synthetic_history(HIST, days_s, currency=cur, seed=1 if cur == 'BTC' else 2,
                             planted=[(int(r.entry_ms), int(r.sign), 40_000.0) for r in sub.itertuples()])
CTL = a6.make_controls(COH, CFG)
ANCH = pd.concat([COH, CTL], ignore_index=True)
print(COH.group.value_counts().to_dict(), '| controls', len(CTL))

## 2. Deribit minute tables (perpetual, dated futures, options) for the needed days

In [ ]:
T0 = time.time(); MIN, COV = {}, []
for cur in ('BTC', 'ETH'):
    a = ANCH[ANCH.currency.eq(cur)]
    if a.empty: continue
    d0 = pd.Timestamp(a.entry_ms.min() - (CFG.window_min + 60) * 60_000, unit='ms', tz='UTC').normalize()
    d1 = pd.Timestamp(a.entry_ms.max() + (CFG.window_min + 60) * 60_000, unit='ms', tz='UTC').normalize()
    days = [str(x.date()) for x in pd.date_range(d0, d1, freq='D')]
    m, cov = a6.build_minutes(HIST, cur, days, log=lambda c, d: print(f'  {c} {d}', flush=True) if d.endswith(('01', '15')) else None)
    MIN[cur] = m; COV.append(cov)
    print(cur, 'minutes', len(m), '| option days', int(cov.option_file.sum()), '/', len(cov), f'| {time.time()-T0:.0f}s')
COV = pd.concat(COV); COV.to_csv(OUT / '06_coverage.csv', index=False)
print('days without future file:', COV.loc[~COV.future_file, ['currency', 'day']].values.tolist()[:10])
print('days without option file:', int((~COV.option_file).sum()))

## 3. Event panel, phases, curves

In [ ]:
PANEL = a6.event_panel(ANCH, MIN, CFG)
RET = COH.set_index('anchor_id').gross_return_bp
PANEL['winloss'] = PANEL.anchor_id.map(lambda a: ('WIN' if RET.get(a, np.nan) > 0 else 'LOSS') if pd.notna(RET.get(a, np.nan)) else None)
PH = a6.phase_panel(PANEL)
PH['winloss'] = PH.anchor_id.map(PANEL.drop_duplicates('anchor_id').set_index('anchor_id').winloss)
CUR = a6.median_curves(PANEL)
num = PANEL.select_dtypes('number').columns
PANEL.assign(**{c: PANEL[c].astype('float32') for c in num if c != 'rel_min'}).to_csv(OUT / '06_event_minute_panel.csv.gz', index=False, float_format='%.5g')
PH.to_csv(OUT / '06_phase_panel.csv', index=False); CUR.to_csv(OUT / '06_median_curves.csv', index=False)
print(PANEL.groupby('group').anchor_id.nunique().to_dict(), '| anchors dropped (window outside data):', len(ANCH) - PANEL.anchor_id.nunique())
display(PH.groupby(['phase', 'group'])[['px_bp', 'imb', 'liq_with', 'liq_against', 'premium_bp', 'atm_iv', 'skew', 'opt_delta_flow', 'fut_basis']].median().round(3))

## 4. Comparisons: TARGET vs STOP (all / near / far target), WIN vs LOSS, SIGNAL vs CONTROL

In [ ]:
PH2 = PH.copy()
sig = PH2[PH2.kind.eq('SIGNAL') & PH2.group.isin(a6.OUTCOMES)].assign(group='SIGNAL')
wl = PH2[PH2.winloss.notna()].assign(group=lambda x: x.winloss)
T = pd.concat([a6.compare(PH2, 'TARGET', 'STOP', s) for s in ('all', 'near_target', 'far_target')] +
              [a6.compare(wl, 'WIN', 'LOSS'), a6.compare(pd.concat([sig, PH2[PH2.group.eq('CONTROL')]]), 'SIGNAL', 'CONTROL')],
              ignore_index=True)
T.to_csv(OUT / '06_tests.csv', index=False)
print('pre-entry differences with q_pre <= 0.10:')
display(T[T.pre_entry & (T.q_pre <= CFG.fdr_q)].sort_values('q_pre').round(4))
print('smallest raw p, pre-entry, TARGET vs STOP (all):')
display(T[(T.A == 'TARGET') & (T.stratum == 'all') & T.pre_entry].sort_values('p').head(10).round(4))
print('entry and post-entry (anatomy only):')
display(T[(T.A == 'TARGET') & (T.stratum == 'all') & ~T.pre_entry].sort_values('p').head(12).round(4))

## 5. What happens exactly at entry: minutes −5 … +5, medians by group

In [ ]:
snap = PANEL[PANEL.rel_min.between(-5, 5)].groupby(['group', 'rel_min'])[['px_bp', 'imb', 'vol_rel', 'liq_with', 'liq_against', 'premium_bp', 'opt_delta_flow']].agg(['median', 'mean'])
snap.to_csv(OUT / '06_entry_snapshot.csv'); display(snap.round(3))

## 6. Verdict file

In [ ]:
DEC = a6.decision(T, PANEL, COV, CFG)
DEC['synthetic'] = SYNTHETIC
(OUT / '06_decision.json').write_text(json.dumps(DEC, indent=1, default=str)); print(json.dumps(DEC, indent=1, default=str))

## 7. Curves (median; liquidation/blocks/option flow as 15-min rolling means across events)

In [ ]:
feats = [('px_bp', 'median'), ('imb', 'median'), ('premium_bp', 'median'), ('liq_with', 'mean15'), ('liq_against', 'mean15'),
         ('opt_delta_flow', 'mean15'), ('atm_iv', 'median'), ('skew', 'median'), ('fut_basis', 'median')]
cols = {'TARGET': 'tab:green', 'STOP': 'tab:red', 'TIMEOUT': 'tab:orange', 'CONTROL': 'grey'}
fig, ax = plt.subplots(3, 3, figsize=(18, 12)); ax = ax.ravel()
for i, (f, how) in enumerate(feats):
    for g, c in cols.items():
        z = PANEL[PANEL.group.eq(g)]
        if z.empty: continue
        if how == 'median':
            y = z.groupby('rel_min')[f].median()
        else:
            y = z.groupby('rel_min')[f].mean().rolling(15, center=True, min_periods=5).mean()
        ax[i].plot(y.index, y.values, c=c, label=g, lw=1.4 if g != 'CONTROL' else 1.0)
    ax[i].axvline(0, c='k', lw=.6); ax[i].axhline(0, c='k', lw=.3); ax[i].set_title(f'{f} ({how})')
ax[0].legend(); plt.tight_layout(); plt.savefig(OUT / '06_context_curves.png', dpi=110); plt.show()